In [11]:
import torch
import torch.nn.functional as F
from ultralytics import YOLO

# =====================================================================
# 1. FUNCIÓN DE PÉRDIDA PERSONALIZADA (KNOWLEDGE DISTILLATION)
# =====================================================================
def make_custom_distillation_loss(native_loss_func, model_teacher, num_classes=12, temperature=4.0, alpha=0.4, beta=0.3):
    """
    Envuelve la pérdida nativa separando limpiamente la destilación de 
    Clasificación (KL Divergence) y la de Bounding Boxes (MSE).
    """
    # 64 canales es el estándar de DFL para cajas en los heads de YOLO (4 coordenadas x 16 bins)
    reg_max_channels = 64 

    def custom_loss(preds, batch):
        # 1. Pérdida original (Hard Loss) contra el Ground Truth real
        loss_hard, loss_items = native_loss_func(preds, batch)

        # 2. Inferencia del Profesor congelado
        with torch.no_grad():
            teacher_preds = model_teacher(batch["img"])

        loss_soft_class = 0.0
        loss_soft_bbox = 0.0

        # Iteramos sobre las distintas escalas de predicción (normalmente 3 capas)
        for s_layer, t_layer in zip(preds, teacher_preds):
            
            # --- SEPARACIÓN DE CANALES ---
            # Ajustamos dimensiones para trabajar cómodamente sobre el eje de canales (dim=1)
            # s_layer shape: [B, channels, H, W] -> permutamos a [B, H, W, channels]
            s_perm = s_layer.permute(0, 2, 3, 1)
            t_perm = t_layer.permute(0, 2, 3, 1)

            # Extraemos los parches correspondientes a Cajas y Clases
            s_boxes = s_perm[..., :reg_max_channels]
            t_boxes = t_perm[..., :reg_max_channels]

            s_classes = s_perm[..., reg_max_channels : reg_max_channels + num_classes]
            t_classes = t_perm[..., reg_max_channels : reg_max_channels + num_classes]

            # --- DESTILACIÓN DE CLASIFICACIÓN (Softmax + KL) ---
            soft_s = F.log_softmax(s_classes / temperature, dim=-1)
            soft_t = F.softmax(t_classes / temperature, dim=-1)
            loss_soft_class += F.kl_div(soft_s, soft_t, reduction="batchmean") * (temperature ** 2)

            # --- DESTILACIÓN DE CAJAS (Regresión continua con MSE) ---
            # El alumno imita directamente los mapas de regresión fina del profesor
            loss_soft_bbox += F.mse_loss(s_boxes, t_boxes, reduction="mean")

        # 3. FUSIÓN TRIPLE DE CONOCIMIENTO
        # alpha: peso del criterio del profesor en clases
        # beta: peso del criterio del profesor en la geometría de las cajas
        total_loss = ((1.0 - alpha - beta) * loss_hard) + (alpha * loss_soft_class) + (beta * loss_soft_bbox)
        
        return total_loss, loss_items

    return custom_loss

# =====================================================================
# 2. SUBCLASE DE YOLO CON INTERCEPCIÓN DINÁMICA PROTEGIDA
# =====================================================================
class DistilledYOLO(YOLO):
    def __init__(self, model="yolo26n.yaml", teacher_model=None, temperature=4.0, alpha=0.4):
        # Primero inicializamos la clase base de Ultralytics
        super().__init__(model)
        
        # Evitamos el __getattr__ de YOLO guardando las variables en __dict__ directamente
        self.__dict__["teacher"] = teacher_model
        self.__dict__["T"] = temperature
        self.__dict__["alpha"] = alpha
        
        # Si pasamos un profesor válido, interceptamos de forma segura
        if self.__dict__["teacher"] is not None:
            self._register_distillation_hook()

    def _register_distillation_hook(self):
        # Accedemos al modelo interno de PyTorch asignado al Alumno y al Profesor
        student_pytorch_model = self.model
        teacher_pytorch_model = self.__dict__["teacher"].model
        teacher_pytorch_model.eval() # El profesor siempre en modo evaluación

        # Guardamos el método original que inicializa la pérdida en Ultralytics
        original_init_criterion = student_pytorch_model.init_criterion

        # Definimos nuestra versión modificada
        def hooked_init_criterion():
            # 1. Dejar que Ultralytics cree su Loss original
            original_init_criterion()
            # 2. Interceptarlo con nuestra lógica de destilación usando las variables del dict
            student_pytorch_model.criterion = make_custom_distillation_loss(
                native_loss_func=student_pytorch_model.criterion,
                model_teacher=teacher_pytorch_model,
                temperature=self.__dict__["T"],
                alpha=self.__dict__["alpha"]
            )
            print("--> [HOOK] Criterio de pérdida de destilación acoplado correctamente.")

        # Reemplazar el método nativo por el nuestro modificado
        student_pytorch_model.init_criterion = hooked_init_criterion

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Ejecutando entorno en: {device}")

# A. Cargamos el Profesor pesado
print("Cargando modelo Profesor (yolo26x)...")
teacher_instance = YOLO("yolo26x.pt").to(device)

# B. Cargamos nuestro Alumno ligero de forma segura
print("Inicializando modelo Alumno (yolo26n)...")
student_instance = DistilledYOLO(
    model="yolo26n.yaml",  
    teacher_model=teacher_instance, 
    temperature=4.0, 
    alpha=0.4
)

# C. Configuración estándar de argumentos para tu entrenamiento
dataset_path = "../datasets/VOC/data.yaml"

train_args = {
    "data": dataset_path,
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "project": "YOLO26_Distillation",
    "name": "yolo26_distilled_stable",
    "device": device
}

# D. Lanzamos el entrenamiento nativo protegido
print("Arrancando bucle de entrenamiento con destilación...")
student_instance.train(**train_args)
print("¡Proceso de destilación completado!")

Ejecutando entorno en: cuda
Cargando modelo Profesor (yolo26x)...
Inicializando modelo Alumno (yolo26n)...
Arrancando bucle de entrenamiento con destilación...
New https://pypi.org/project/ultralytics/8.4.52 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24126MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/VOC/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, l

Exception in thread Thread-38 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
Exception ignored in: <function _releaseLock at 0x7a737d781760>
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/josericardopenase/Escritorio/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    r

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000417, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
: 0% ──────────── 0/715  9.5s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
: 0% ──────────── 0/1430  0.0s


RuntimeError: Pin memory thread exited unexpectedly

In [8]:
from ultralytics import YOLO
import torch
from ultralytics.nn.tasks import DetectionModel
import pandas as pd
import pandas as pd

dataset = 'VOC'
yolo = 'yolo26m'
experiment='(5750:5000,Uniform)'

# load results
df = pd.read_csv(f"../runs/detect/{yolo}{dataset}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
baseline = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}

print(baseline)

df = pd.read_csv(f"../runs/detect/{yolo}{dataset}_{experiment}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
exp = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}


print(exp)
improvement_pct = {k: ((exp[k] - baseline[k]) / baseline[k] * 100) if baseline[k] != 0 else float('inf') for k in exp}
print(improvement_pct)

model_exp  = YOLO(f'../runs/detect/{yolo}{dataset}_{experiment}/weights/best.pt')

model_baseline = YOLO(f'../runs/detect/{yolo}{dataset}/weights/best.pt')

metrics_baseline = model_baseline.val(data="../datasets/kitti/kitti.yaml", imgsz=640, device=1, name='./evals/yolo26nKITTI_(5750:5000,Uniform)',  verbose=False)
metrics_exp = model_exp.val(data="../datasets/kitti/kitti.yaml", imgsz=640, device=1, name='./evals/yolo26nKITTI_(5750:5000,Uniform)',  verbose=False)

# Get class names and maps values
classes = list(metrics_baseline.names.values() if hasattr(metrics_baseline, "names") else metrics_baseline.names)
baseline_maps = metrics_baseline.maps if hasattr(metrics_baseline, "maps") else metrics_baseline.results_dict['maps']
exp_maps = metrics_exp.maps if hasattr(metrics_exp, "maps") else metrics_exp.results_dict['maps']

# Build the DataFrame
comparative_df = pd.DataFrame({
    "Class": classes,
    "Baseline mAP50-95": baseline_maps,
    "Experiment mAP50-95": exp_maps,
    "Improvement (%)": ((exp_maps - baseline_maps) / baseline_maps * 100)
})



{'precision': np.float64(0.84368), 'recall': np.float64(0.68428), 'mAP50': np.float64(0.78013), 'mAP50-95': np.float64(0.62593)}
{'precision': np.float64(0.84126), 'recall': np.float64(0.71624), 'mAP50': np.float64(0.79643), 'mAP50-95': np.float64(0.64388)}
{'precision': np.float64(-0.28683861179593895), 'recall': np.float64(4.670602677266614), 'mAP50': np.float64(2.0893953571840567), 'mAP50-95': np.float64(2.867732813573406)}
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:1 (NVIDIA GeForce RTX 4060 Ti, 15948MiB)
YOLO26m summary (fused): 132 layers, 20,364,872 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1751.9±600.6 MB/s, size: 43.4 KB)
val: Scanning /home/josericardopenase/Escritorio/datasets/kitti/labels/val.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 570.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 94/94 9.8it/s 9.6s<0.1s
 

/tmp/ipykernel_2298/1644445836.py:62: RuntimeWarning: invalid value encountered in divide
  "Improvement (%)": ((exp_maps - baseline_maps) / baseline_maps * 100)


In [10]:
# Display improvement_pct as a formatted DataFrame
improvement_df = pd.DataFrame(list(improvement_pct.items()), columns=["Metric", "Improvement (%)"])
print(improvement_df.to_string(index=False))

comparative_df

   Metric  Improvement (%)
precision        -0.286839
   recall         4.670603
    mAP50         2.089395
 mAP50-95         2.867733


,Class,Baseline mAP50-95,Experiment mAP50-95,Improvement (%)
0,aeroplane,0.075763,0.004056,-94.646298
1,bicycle,0.000000,0.000000,NaN
2,bird,0.000000,0.000000,NaN
3,boat,0.000000,0.000000,NaN
4,bottle,0.000000,0.000000,NaN
5,bus,0.000000,0.000000,NaN
6,car,0.000674,0.000614,-8.841372
7,cat,0.000000,0.000000,NaN
8,chair,0.009555,0.000584,-93.890231
9,cow,0.009555,0.000584,-93.890231


In [8]:
dataset = 'KITTI'
yolo = 'yolo26x'
experiment='(5750:5000,Uniform)'

# load results
df = pd.read_csv(f"../runs/detect/{yolo}{dataset}/results.csv")

# choose best model by mAP50-95
best_row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]

# extract metrics
baseline = {
    "precision": best_row["metrics/precision(B)"],
    "recall": best_row["metrics/recall(B)"],
    "mAP50": best_row["metrics/mAP50(B)"],
    "mAP50-95": best_row["metrics/mAP50-95(B)"],
}
baseline

{'precision': np.float64(0.64756),
 'recall': np.float64(0.54747),
 'mAP50': np.float64(0.61553),
 'mAP50-95': np.float64(0.37757)}

In [1]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="african-wildlife.yaml", epochs=100, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.43 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24126MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=african-wildlife.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train,

KeyboardInterrupt: 

In [2]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO("yolo26n.pt")

# Train the model on HomeObjects-3K dataset
model.train(data="HomeObjects-3K.yaml", epochs=100, imgsz=640)

New https://pypi.org/project/ultralytics/8.4.43 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24126MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, 

KeyboardInterrupt: 